# Positional Encoding from scratch

In [9]:
import numpy as np
import matplotlib.pyplot as plt

## 1) Let's go through an example

Self-attention is a very good tool for comparing vectors with other vectors. However vectors by themselves do not contain position unless we put it in somehow. 

In [10]:
tokens = ["the", "cat", "sat", "down"]

for i, tok in enumerate(tokens):
    print(f"position: {i}, token: {tok}")

position: 0, token: the
position: 1, token: cat
position: 2, token: sat
position: 3, token: down


A sequence has two components, its content and its position. The content tells us what the token is while the position tells us where the token sits. In our case we have:

- The token "the" has the content "the" and the position O
- The token "cat" has the content "cat" and the position 1
- The token "sat" has the content "sat" and the position 2
- The token "down" has the content "down" and the position 3

A model that sees only the content has no chance of automatically knowing the order. 

Now we are going to map our tokens to embeddings (vectors) but without a position yet. 

## 2) Representing our tokens as vectors but without position yet

In [11]:
token_2_vec = {

    "the": np.array([1.0, 0.0, 0.0, 0.0]),
    "cat": np.array([0.0, 1.0, 0.0, 0.0]),
    "sat": np.array([0.0, 0.0, 1.0, 0.0]),
    "down": np.array([0.0, 0.0, 0.0, 1.0]),
}

X = np.stack([token_2_vec[t] for t in tokens])
print(f"X: {X}, shape: {X.shape}")

X: [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]], shape: (4, 4)


So now each token is represented as a vector. However each of these vectors now tell us that one vector is corresponding to a meaning. For example [0.0, 1.0, 0.0, 0.0] corresponds to "cat" but it does not tell us whether cat came first or second. We need to encode its position.

## 3) Attention without position cannot distinguish some reorderings

Let's compare two sequences with the same words but with a different ordering

In [13]:
tokens_a = ["the", "cat", "sat", "down"]
tokens_b = ["down", "cat", "the", "sat"]

Xa = np.stack([token_2_vec[t] for t in tokens_a])
Xb = np.stack([token_2_vec[t] for t in tokens_b])

print(f"Xa: {Xa}, shape of Xa: {Xa.shape}")
print(f"Xb: {Xb}, shape of Xb: {Xb.shape}")

Xa: [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]], shape of Xa: (4, 4)
Xb: [[0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]], shape of Xb: (4, 4)


These matrices are of the same shape but they differ by row order. A self-attention layer without positonal encoding is permutation equivariant. What this translates to is that if you permute the input rows then the output rows get permuted the exact same way. The attention mechanism does not have any built-in notion of "left to right order". It reacts to relationships among vectors. 

When you look at the first vector you tell yourself "this is the first vector", but there is nothing in the vector itself that tells you if it is the first or second one and the attention mechanism therefore has no clue of the vector's position. It just sees a bag of vectors. That is why we need positional encoding. 

## 4) A tiny self attention without positional encoding

Let's first define a softmax function

In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True) # we do this for numerical stability since exponential can grow large
    exp_x = np.exp(x) # we compute the exponential of our input vector in order to get back positive values and turn scores into weights
    return exp_x / sum(exp_x, axis=axis, keepdims=True) # we return the normalized result (each exponential is returned divided by the total sum)